# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [68]:
# Load the raw data. latin1 encoding handles non-UTF8 characters (e.g. in
# Location names); low_memory=False avoids dtype-guessing issues on this
# large, mixed-type CSV.
df = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)

print("Shape:", df.shape)
df.info()


Shape: (88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make               

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [69]:
#removing all data before 1983
data = df[df['Event.Date'] >= '1983-01-01']

In [70]:
# --- Inspect columns relevant to the client's scope ---
print("\nAmateur.Built:\n", data['Amateur.Built'].value_counts(dropna=False))
print("\nAircraft.Category:\n", data['Aircraft.Category'].value_counts(dropna=False))



Amateur.Built:
 Amateur.Built
No     76960
Yes     8228
NaN      101
Name: count, dtype: int64

Aircraft.Category:
 Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


In [71]:
 #--- Filtering decisions ---

# Client wants "professional builds" only
data = data[data['Amateur.Built'] == 'No']

# Client is only interested in airplanes
data = data[data['Aircraft.Category'] == 'Airplane']

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21447 entries, 4149 to 88886
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                21447 non-null  object 
 1   Investigation.Type      21447 non-null  object 
 2   Accident.Number         21447 non-null  object 
 3   Event.Date              21447 non-null  object 
 4   Location                21441 non-null  object 
 5   Country                 21446 non-null  object 
 6   Latitude                19169 non-null  object 
 7   Longitude               19163 non-null  object 
 8   Airport.Code            13983 non-null  object 
 9   Airport.Name            14070 non-null  object 
 10  Injury.Severity         20634 non-null  object 
 11  Aircraft.damage         20220 non-null  object 
 12  Aircraft.Category       21447 non-null  object 
 13  Registration.Number     21241 non-null  object 
 14  Make                    21444 non-null  

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

Assumptions based of research on NTSB:

- NTSB tends to omit a field rather than write 0


In [72]:
inj_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
            'Total.Minor.Injuries', 'Total.Uninjured']

# NaN in these columns most likely means "0 of that category", so fill NaN with 0 before summing.
data[inj_cols] = data[inj_cols].fillna(0)

# Estimated total people on board = sum across all 4 outcome buckets
data['total_onboard'] = data[inj_cols].sum(axis=1)

# Can't compute a rate for flights where we have no info on anyone at all
data = data[data['total_onboard'] > 0]

# The metric itself: fraction of people on board who were seriously or fatally hurt
data['serious_fatal_fraction'] = (data['Total.Fatal.Injuries'] + data['Total.Serious.Injuries']) / data['total_onboard']

In [73]:
print(data.shape)

(20543, 33)


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [74]:
print(data['Aircraft.damage'].value_counts(dropna=False))

# Dropped the small number of rows where damage severity is missing or 'Unknown'
# this is our other key target variable, and we can't infer it.
data = data[data['Aircraft.damage'].isin(['Substantial', 'Destroyed', 'Minor'])]

# Derived binary column: was the aircraft a total loss ("Destroyed") or not
# (Substantial/Minor damage)? This is the client's core "total destruction"
# outcome of interest.
data['Destroyed'] = (data['Aircraft.damage'] == 'Destroyed').astype(int)

print("\nShape:", data.shape)
data['Destroyed'].value_counts(normalize=True)
#11% destroyed, 88% not destroyed

Aircraft.damage
Substantial    16821
Destroyed       2268
NaN              787
Minor            593
Unknown           74
Name: count, dtype: int64

Shape: (19682, 34)


,proportion
Destroyed,
0,0.884768
1,0.115232


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [75]:
print("Unique raw Make values:", data['Make'].nunique())
print(data['Make'].value_counts().head(15))

Unique raw Make values: 1289
Make
CESSNA                4694
PIPER                 2740
Cessna                2252
Piper                 1177
BEECH                  987
Beech                  405
BOEING                 363
MOONEY                 230
AIR TRACTOR INC        216
CIRRUS DESIGN CORP     214
BELLANCA               157
AERONCA                149
MAULE                  144
Mooney                 124
Boeing                 118
Name: count, dtype: int64


In [76]:
# Missing Values need to be dropped because you can't estimate the make
data = data.dropna(subset=['Make'])
# Inconsistent casing -- e.g. 'Cessna' vs 'CESSNA' are the same
# manufacturer, split into two separate value_counts buckets.
# Leading/trailing whitespace.
# Fix: strip whitespace and title-case everything.
data['Make'] = data['Make'].str.strip().str.title()

print("\nUnique Make values after normalizing case:", data['Make'].nunique())
print(data.shape)
print(data['Make'].value_counts().head(15))


Unique Make values after normalizing case: 1049
(19681, 34)
Make
Cessna                6946
Piper                 3917
Beech                 1392
Boeing                 481
Mooney                 354
Bellanca               218
Air Tractor Inc        218
Cirrus Design Corp     216
Maule                  215
Air Tractor            203
Aeronca                200
Champion               157
Grumman                145
Luscombe               138
Stinson                129
Name: count, dtype: int64


In [77]:
# For statistically robust make-level comparisons, only keep makes with a
# reasonable sample size (threshold: at least 50 accident records).
make_counts = data['Make'].value_counts()
keep_makes = make_counts[make_counts >= 50].index
print(f"\nMakes with >=50 records: {len(keep_makes)} (of {data['Make'].nunique()} total)")

data = data[data['Make'].isin(keep_makes)]
print("Shape after Make filtering:", data.shape)
print(data['Make'].value_counts().head(15))


Makes with >=50 records: 34 (of 1049 total)
Shape after Make filtering: (16320, 34)
Make
Cessna                6946
Piper                 3917
Beech                 1392
Boeing                 481
Mooney                 354
Bellanca               218
Air Tractor Inc        218
Cirrus Design Corp     216
Maule                  215
Air Tractor            203
Aeronca                200
Champion               157
Grumman                145
Luscombe               138
Stinson                129
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [78]:
print("Model NaNs:", data['Model'].isna().sum())
data = data.dropna(subset=['Model'])

Model NaNs: 10


In [79]:
data.shape

(16310, 34)

In [80]:
# Normalize casing/whitespace
data['Model'] = data['Model'].str.strip().str.upper()

In [81]:
# Are model labels unique to a single make? Check how many model strings
# appear under more than one manufacturer.
model_make_map = data.groupby('Model')['Make'].nunique()
shared = (model_make_map > 1).sum()
print(f"\nModel labels shared across multiple makes: {shared} of {len(model_make_map)}")
print(model_make_map[model_make_map > 1].head(10))


Model labels shared across multiple makes: 93 of 1794
Model
100      2
112      2
112A     2
1900D    2
200      2
320      2
350      2
390      2
400      2
400A     2
Name: Make, dtype: int64


In [82]:
# Model labels are NOT unique per make, so we build a combined Make_Model identifier to uniquely
# identify a specific airplane type.
data['Make_Model'] = data['Make'] + ' ' + data['Model']
print("\nUnique Make_Model combinations:", data['Make_Model'].nunique())
print("Shape:", data.shape)


Unique Make_Model combinations: 1896
Shape: (16310, 35)


In [83]:
data.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,FAR.Description,Schedule,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,total_onboard,serious_fatal_fraction,Destroyed,Make_Model
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,Incident,Minor,Airplane,9VSQQ,Boeing,747,No,4.0,Turbo Fan,Part 129: Foreign,SCHD,NaN,"Singapore Airlines, Ltd.",0.0,0.0,0.0,588.0,VMC,Taxi,Probable Cause,04-12-2014,588.0,0.0,0,Boeing 747
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,Fatal(1),Destroyed,Airplane,N9600W,Piper,PA-28-140,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,1.0,1.0,0.0,0.0,IMC,Cruise,Probable Cause,02-05-2011,2.0,1.0,1,Piper PA-28-140
6760,20001214X45013,Incident,CHI84IA041,1983-11-08,"CHICAGO, IL",United States,NaN,NaN,ORD,O'HARE,Incident,Minor,Airplane,N898AA,Boeing,727-200,No,3.0,Turbo Fan,Part 121: Air Carrier,SCHD,Unknown,NaN,0.0,0.0,0.0,100.0,VMC,Taxi,Probable Cause,11-06-2018,100.0,0.0,0,Boeing 727-200
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,NaN,NaN,NaN,NaN,Non-Fatal,Substantial,Airplane,N1882D,Beech,C35,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,0.0,0.0,0.0,1.0,VMC,Climb,Probable Cause,05-05-2011,1.0,0.0,0,Beech C35
7084,20001214X45339,Accident,LAX84LA110,1983-12-22,"SANTA ROSA ISLAND, CA",United States,NaN,NaN,NaN,PRIVATE,Non-Fatal,Substantial,Airplane,N2697K,Cessna,180K,No,1.0,Reciprocating,Part 91: General Aviation,NaN,Personal,NaN,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,01-02-2016,1.0,0.0,0,Cessna 180K


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

In [84]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
            'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False))
    print()

--- Engine.Type ---
Engine.Type
Reciprocating      12745
NaN                 2247
Turbo Prop           882
Turbo Fan            321
Unknown               64
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

--- Weather.Condition ---
Weather.Condition
VMC    13916
NaN     1381
IMC      844
Unk      128
UNK       41
Name: count, dtype: int64

--- Number.of.Engines ---
Number.of.Engines
1.0    13081
2.0     1885
NaN     1289
4.0       36
3.0       16
0.0        3
Name: count, dtype: int64

--- Purpose.of.flight ---
Purpose.of.flight
Personal                     9771
Instructional                2378
NaN                          1692
Aerial Application            721
Business                      399
Positioning                   255
Unknown                       244
Skydiving                     153
Aerial Observation            145
Other Work Use                117
Banner Tow                     86
Ferry        

In [85]:
# Consolidate duplicate spellings of the same category
data['Engine.Type'] = data['Engine.Type'].replace({'UNK': 'Unknown'})
data['Weather.Condition'] = data['Weather.Condition'].replace({'Unk': 'UNK'})
data['Purpose.of.flight'] = data['Purpose.of.flight'].replace({
    'Air Race show': 'Air Race/Show',
    'Air Race/show': 'Air Race/Show'
})

In [86]:
# Number.of.Engines == 0 doesn't make physical sense for a powered airplane
# (confirmed via spot-check: e.g. Weatherly 201B is a real single-engine
# crop duster, so its 0 here is a data-entry error) -> treat as missing
data['Number.of.Engines'] = data['Number.of.Engines'].replace({0: np.nan})

In [87]:
# 'Unknown'/'UNK' are placeholders for missing data, not real categories
# -> convert to genuine NaN so they don't get treated as a valid group
data['Engine.Type'] = data['Engine.Type'].replace({'Unknown': np.nan})
data['Weather.Condition'] = data['Weather.Condition'].replace({'UNK': np.nan})
data['Purpose.of.flight'] = data['Purpose.of.flight'].replace({'Unknown': np.nan})
data['Broad.phase.of.flight'] = data['Broad.phase.of.flight'].replace({'Unknown': np.nan})

In [88]:
categorical_cols = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
    "Number.of.Engines"
]
data[categorical_cols] = data[categorical_cols].fillna("Unknown")

In [89]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
            'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False))
    print()

--- Engine.Type ---
Engine.Type
Reciprocating      12745
Unknown             2312
Turbo Prop           882
Turbo Fan            321
Turbo Jet             41
Turbo Shaft            8
Geared Turbofan        1
Name: count, dtype: int64

--- Weather.Condition ---
Weather.Condition
VMC        13916
Unknown     1550
IMC          844
Name: count, dtype: int64

--- Number.of.Engines ---
Number.of.Engines
1.0        13081
2.0         1885
Unknown     1292
4.0           36
3.0           16
Name: count, dtype: int64

--- Purpose.of.flight ---
Purpose.of.flight
Personal                     9771
Instructional                2378
Unknown                      1936
Aerial Application            721
Business                      399
Positioning                   255
Skydiving                     153
Aerial Observation            145
Other Work Use                117
Banner Tow                     86
Ferry                          71
Flight Test                    70
Executive/corporate            62
Gl

In [90]:
# Purpose.of.flight
# Categories representing less than 1%
# of the dataset into 'Other'

purpose_counts = data["Purpose.of.flight"].value_counts()

# Calculate 1% of the dataset
purpose_threshold = len(data) * 0.01

# Identify rare categories, excluding meaningful categories
rare_purposes = purpose_counts[
    (purpose_counts < purpose_threshold) &
    (~purpose_counts.index.isin(["Unknown", "Skydiving"]))
].index

# Replace rare categories with 'Other'
data["Purpose.of.flight"] = data["Purpose.of.flight"].replace(
    dict.fromkeys(rare_purposes, "Other")
)

print("1% threshold:", purpose_threshold)
print("\nCleaned Purpose.of.flight categories:")
print(data["Purpose.of.flight"].value_counts())

print("\nCleaned up. Shape:", data.shape)

1% threshold: 163.1

Cleaned Purpose.of.flight categories:
Purpose.of.flight
Personal              9771
Instructional         2378
Unknown               1936
Aerial Application     721
Other                  697
Business               399
Positioning            255
Skydiving              153
Name: count, dtype: int64

Cleaned up. Shape: (16310, 35)


In [91]:
# -----------------------------------------
# Number.of.Engines
# Group 3 and 4 engines as "3+"
# -----------------------------------------

data["Number.of.Engines"] = data["Number.of.Engines"].replace({
    1.0: "1",
    2.0: "2",
    3.0: "3+",
    4.0: "3+",
    "1.0": "1",
    "2.0": "2",
    "3.0": "3+",
    "4.0": "3+"
})

In [92]:
# -----------------------------------------
# Engine.Type
# Group categories with fewer than 50 rows
# -----------------------------------------

engine_counts = data["Engine.Type"].value_counts()

rare_engines = engine_counts[
    (engine_counts < 50) &
    (engine_counts.index != "Unknown")
].index

data["Engine.Type"] = data["Engine.Type"].replace(
    rare_engines,
    "Other"
)

In [93]:
# -----------------------------------------
# Broad.phase.of.flight
# Group selected rare phases using 1% of data Count threshold
# -----------------------------------------

rare_flight_phases = [
    "Descent",
    "Climb",
    "Standing",
    "Other"

]

data["Broad.phase.of.flight"] = data[
    "Broad.phase.of.flight"
].replace(rare_flight_phases, "Other")


In [94]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
            'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False))
    print()

--- Engine.Type ---
Engine.Type
Reciprocating    12745
Unknown           2312
Turbo Prop         882
Turbo Fan          321
Other               50
Name: count, dtype: int64

--- Weather.Condition ---
Weather.Condition
VMC        13916
Unknown     1550
IMC          844
Name: count, dtype: int64

--- Number.of.Engines ---
Number.of.Engines
1          13081
2           1885
Unknown     1292
3+            52
Name: count, dtype: int64

--- Purpose.of.flight ---
Purpose.of.flight
Personal              9771
Instructional         2378
Unknown               1936
Aerial Application     721
Other                  697
Business               399
Positioning            255
Skydiving              153
Name: count, dtype: int64

--- Broad.phase.of.flight ---
Broad.phase.of.flight
Unknown        13901
Landing         1108
Takeoff          423
Cruise           232
Approach         208
Other            136
Maneuvering      127
Taxi              94
Go-around         81
Name: count, dtype: int64



### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [95]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16310 entries, 4150 to 88886
Data columns (total 35 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                16310 non-null  object 
 1   Investigation.Type      16310 non-null  object 
 2   Accident.Number         16310 non-null  object 
 3   Event.Date              16310 non-null  object 
 4   Location                16307 non-null  object 
 5   Country                 16309 non-null  object 
 6   Latitude                15317 non-null  object 
 7   Longitude               15313 non-null  object 
 8   Airport.Code            11123 non-null  object 
 9   Airport.Name            11229 non-null  object 
 10  Injury.Severity         16310 non-null  object 
 11  Aircraft.damage         16310 non-null  object 
 12  Aircraft.Category       16310 non-null  object 
 13  Registration.Number     16193 non-null  object 
 14  Make                    16310 non-null  

In [96]:
# Recheck NaN levels now that the row set has been filtered down.
na_counts = data.isna().sum().sort_values(ascending=False)
print(na_counts)

Schedule                  15027
Air.carrier                8836
Airport.Code               5187
Airport.Name               5081
Report.Status              2746
Longitude                   997
Latitude                    993
Publication.Date            494
FAR.Description             157
Registration.Number         117
Location                      3
Country                       1
Accident.Number               0
Event.Id                      0
Investigation.Type            0
Injury.Severity               0
Aircraft.damage               0
Aircraft.Category             0
Event.Date                    0
Engine.Type                   0
Number.of.Engines             0
Amateur.Built                 0
Model                         0
Make                          0
Total.Serious.Injuries        0
Total.Fatal.Injuries          0
Purpose.of.flight             0
Total.Uninjured               0
Total.Minor.Injuries          0
Broad.phase.of.flight         0
Weather.Condition             0
total_on

In [97]:
columns_to_drop = [
    "Schedule",
    "Air.carrier",
    "Airport.Code",
    "Airport.Name",
    "Report.Status",
    "Publication.Date",
    "Longitude",
    "Latitude",
    "Registration.Number",
]
aviation_data_clean = data.drop(columns=columns_to_drop)
print(aviation_data_clean.shape)

(16310, 26)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [98]:
aviation_data_clean.to_csv('aviation_data_clean.csv', index=False)

print("Saved cleaned data:", aviation_data_clean.shape)

Saved cleaned data: (16310, 26)
